In [ ]:
from Bio import SeqIO
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import gzip

def reverse_complement(dna_seq):
    """ 
    Return the reverse complement of a DNA sequence. 
    Arguments:
        dna_seq - DNA input sequence as a string
        
    Returns:
        reverse_complement - Reverse complement of DNA input sequence 
    
    """
    complement = {'a': 't', 'A':'T', 't': 'a', 'T':'A','c': 'g','C':'G', 'g': 'c', 'G':'C'}
    
    reverse_complement_seq = ''.join(complement.get(base, base) for base in reversed(dna_seq))
        
    return reverse_complement_seq

In [ ]:
output_name = 'test_output' #Name of output file, will be saved as .test in the same directory as this notebook. Do not include .csv in the name, it will be added automatically.
rev_comp = False #True/False, set to True if the sequence of interest is on the reverse strand
variant_codon = 'TEST' #Insert the codon of interest in the format 'AAA' (3 nucleotides, no spaces)


#Insert path to fastq file below 
fastq_file =""
print('FastQ file:')
print(fastq_file)

#prefix of desired amplicon up to codon of interest
prefix =""
prefix = prefix.upper()
print('Prefix:')
print(prefix)

#suffix of desired amplicon from codon of interest to end of read length
suffix = ""
suffix = suffix.upper()
print('Suffix')
print(suffix)

In [ ]:
# Collect sequence lengths and plot histogram
with gzip.open(fastq_file, "rt") as handle:
    for record in SeqIO.parse(handle, "fastq"):
        lengths = [len(record.seq)]

# Plot histogram
plt.hist(lengths, bins=50, edgecolor="black")
plt.xlabel("Sequence Length")
plt.ylabel("Count")
plt.title("Distribution of Sequence Lengths")
plt.show()

In [ ]:
# Parse the FASTQ file and count sequences
seq_counter = Counter()

with gzip.open(fastq_file, "rt") as handle:
    for record in SeqIO.parse(handle, "fastq"):
        seq_str = str(record.seq)
        seq_counter[seq_str] += 1

# Print the 1000 most common sequences
for seq, count in seq_counter.most_common(1000):
    print(f"{seq} : {count}")

In [ ]:
import regex
if rev_comp == True:
    target_sequences = {
        r'.*':0, # All reads
        prefix +'[ATCG]{3}'+ suffix:0,#All variants
        prefix + reverse_complement('GGC')+ suffix:0, #A GCC
        prefix + reverse_complement('TGC')+ suffix:0, #C TGC
        prefix + reverse_complement('GAC')+ suffix:0, #D GAC
        prefix + reverse_complement('GAG')+ suffix:0, #E GAG
        prefix + reverse_complement('TTC')+ suffix:0, #F TTC
        prefix + reverse_complement('GCC')+ suffix:0, #G GGC
        prefix + reverse_complement('CAC')+ suffix:0, #H CAC
        prefix + reverse_complement('ATC')+ suffix:0, #I ATC
        prefix + reverse_complement('AAG')+ suffix:0, #K AAG
        prefix + reverse_complement('CTG')+ suffix:0, #L CTG
        prefix + reverse_complement('ATG')+ suffix:0, #M ATG
        prefix + reverse_complement('AAC')+ suffix:0, #N AAC
        prefix + reverse_complement('CCC')+ suffix:0, #P CCC
        prefix + reverse_complement('CAG')+ suffix:0, #Q CAG
        prefix + reverse_complement('CGG')+ suffix:0, #R CGG
        prefix + reverse_complement('AGC')+ suffix:0, #S AGC
        prefix + reverse_complement('ACC')+ suffix:0, #T ACC
        prefix + reverse_complement('GTG')+ suffix:0, #V GTG
        prefix + reverse_complement('TGG')+ suffix:0, #W TGG
        prefix + reverse_complement('TAC')+ suffix:0, #Y TAC
        prefix + reverse_complement('TAA')+ suffix:0, #STOP TAA
        prefix + reverse_complement(variant_codon)+ suffix:0, #Patient Variant
        }
       
else:
    target_sequences = {
        r'.*':0, # All reads
        prefix +'[ATCG]{3}'+ suffix:0,#All variants
        prefix +'GCC'+ suffix:0, #A GCC
        prefix +'TGC'+ suffix:0, #C TGC
        prefix +'GAC'+ suffix:0, #D GAC
        prefix +'GAG'+ suffix:0, #E GAG
        prefix +'TTC'+ suffix:0, #F TTC
        prefix +'GGC'+ suffix:0, #G GGC
        prefix +'CAC'+ suffix:0, #H CAC
        prefix +'ATC'+ suffix:0, #I ATC
        prefix +'AAG'+ suffix:0, #K AAG
        prefix +'CTG'+ suffix:0, #L CTG
        prefix +'ATG'+ suffix:0, #M ATG
        prefix +'AAC'+ suffix:0, #N AAC
        prefix +'CCC'+ suffix:0, #P CCC
        prefix +'CAG'+ suffix:0, #Q CAG
        prefix +'CGG'+ suffix:0, #R CGG
        prefix +'AGC'+ suffix:0, #S AGC
        prefix +'ACC'+ suffix:0, #T ACC
        prefix +'GTG'+ suffix:0, #V GTG
        prefix +'TGG'+ suffix:0, #W TGG
        prefix +'TAC'+ suffix:0, #Y TAC
        prefix +'TAA'+ suffix:0, #STOP TAA
        prefix + variant_codon+ suffix:0, #Patient Variant
        }

# Pre-compile regex for efficiency, but mark exact matches
compiled_targets = {
    pattern: regex.compile(pattern) for pattern in target_sequences
}

target_counts = Counter(target_sequences)

# Parse FASTQ
with gzip.open(fastq_file, "rt") as handle:
    for record in SeqIO.parse(handle, "fastq"):
        seq = str(record.seq)
        for pattern, regex in compiled_targets.items():
            if regex.search(seq):  # regex search 
                target_counts[pattern] += 1

# Build DataFrame
df = pd.DataFrame(list(target_counts.items()), columns=["Sequence/Pattern", "Counts"])
# Add a column for labels
df['label'] = ['All reads', 'All codons', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K' , 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'Stop', 'Patient Variant' ]
display(df)

df.to_excel (output_name + '.xlsx', index=True)
